In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

In [2]:
np.random.seed(1)

In [3]:
# Simulation parameters

n_schools = 5
n_classes_per_school = 4
n_students_per_class = 20
n_sessions = 6

population_intercept = 80
population_slope = 2.5

school_sd = 5
class_sd = 3
student_intercept_sd = 6
student_slope_sd = 0.8
residual_sd = 2

In [4]:
# Simulating hierarchical dataset.

data = []

student_id = 0

for school in range(n_schools):

    school_effect = np.random.normal(
        0,
        school_sd,
    )

    for classroom in range(n_classes_per_school):

        class_effect = np.random.normal(
            0,
            class_sd,
        )

        class_id = school * n_classes_per_school + classroom

        for student in range(n_students_per_class):

            intercept = (
                population_intercept
                + school_effect
                + class_effect
                + np.random.normal(
                    0,
                    student_intercept_sd,
                )
            )

            slope = (
                population_slope
                + np.random.normal(
                    0,
                    student_slope_sd,
                )
            )

            for session in range(1, n_sessions + 1):

                score = (
                    intercept
                    + slope * session
                    + np.random.normal(
                        0,
                        residual_sd,
                    )
                )

                data.append(
                    [
                        school,
                        class_id,
                        student_id,
                        session,
                        score,
                    ]
                )

            student_id += 1

advanced_data = pd.DataFrame(
    data,
    columns=[
        "school",
        "classroom",
        "student",
        "session",
        "score",
    ],
)

In [5]:
# Summary statistics

print("Dataset\n", advanced_data.describe())

print("\nSchools:", advanced_data["school"].nunique(),)

print("Classes:", advanced_data["classroom"].nunique(),)

print("Students:", advanced_data["student"].nunique(),)

Dataset
             school    classroom      student      session        score
count  2400.000000  2400.000000  2400.000000  2400.000000  2400.000000
mean      2.000000     9.500000   199.500000     3.500000    91.961839
std       1.414508     5.767483   115.493757     1.708181    10.639143
min       0.000000     0.000000     0.000000     1.000000    56.444733
25%       1.000000     4.750000    99.750000     2.000000    84.584282
50%       2.000000     9.500000   199.500000     3.500000    91.988015
75%       3.000000    14.250000   299.250000     5.000000    98.810497
max       4.000000    19.000000   399.000000     6.000000   136.034685

Schools: 5
Classes: 20
Students: 400


In [7]:
# Model 1

model_1 = smf.mixedlm(
    "score ~ session",
    data=advanced_data,
    groups="student",
)

results_1 = model_1.fit()

print("\nModel 1: Random intercept\n", results_1.summary())


Model 1: Random intercept
          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: score     
No. Observations: 2400    Method:             REML      
No. Groups:       400     Scale:              6.2166    
Min. group size:  6       Log-Likelihood:     -6490.6057
Max. group size:  6       Converged:          Yes       
Mean group size:  6.0                                   
--------------------------------------------------------
             Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept    83.029    0.484 171.697 0.000 82.081 83.977
session       2.552    0.030  85.641 0.000  2.494  2.611
student Var  88.153    2.774                            



In [8]:
# Model 2

model_2 = smf.mixedlm(
    "score ~ session",
    data=advanced_data,
    groups="student",
    re_formula="~session",
)

results_2 = model_2.fit()

print("\nModel 2: Random intercept and slope\n", results_2.summary())

C:\Users\Erfan\miniforge3\envs\cog-modeling\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Erfan\miniforge3\envs\cog-modeling\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
C:\Users\Erfan\miniforge3\envs\cog-modeling\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Erfan\miniforge3\envs\cog-modeling\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(



Model 2: Random intercept and slope
               Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    score     
No. Observations:    2400       Method:                REML      
No. Groups:          400        Scale:                 5.3741    
Min. group size:     6          Log-Likelihood:        -6355.6728
Max. group size:     6          Converged:             No        
Mean group size:     6.0                                         
-----------------------------------------------------------------
                      Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept             83.029    0.362 229.550 0.000 82.320 83.738
session                2.552    0.037  68.686 0.000  2.479  2.625
student Var           47.674    1.283                            
student x session Cov  0.961    0.079                            
session Var            0.245    0.013               

C:\Users\Erfan\miniforge3\envs\cog-modeling\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Erfan\miniforge3\envs\cog-modeling\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\Erfan\miniforge3\envs\cog-modeling\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 530.752699
  warnings.warn(msg, ConvergenceWarning)


In [9]:
# Model 3

model_3 = smf.mixedlm(
    "score ~ session",
    data=advanced_data,
    groups="classroom",
)

results_3 = model_3.fit()

print("\nModel 3: Classroom random intercept\n", results_3.summary())


Model 3: Classroom random intercept
          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: score     
No. Observations: 2400    Method:             REML      
No. Groups:       20      Scale:              47.1377   
Min. group size:  120     Log-Likelihood:     -8076.7919
Max. group size:  120     Converged:          Yes       
Mean group size:  120.0                                 
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept     83.029    1.606 51.710 0.000 79.882 86.176
session        2.552    0.082 31.101 0.000  2.391  2.713
classroom Var 49.520    2.368                           



In [11]:
# Model Comparison

comparison = pd.DataFrame(
    {
        "Model": [
            "Random intercept",
            "Random intercept + slope",
            "Classroom intercept",
        ],
        "AIC": [
            results_1.aic,
            results_2.aic,
            results_3.aic,
        ],
        "BIC": [
            results_1.bic,
            results_2.bic,
            results_3.bic,
        ],
        "Log Likelihood": [
            results_1.llf,
            results_2.llf,
            results_3.llf,
        ],
    }
)

print("\nModel comparison\n", comparison)


Model comparison
                       Model  AIC  BIC  Log Likelihood
0          Random intercept  NaN  NaN    -6490.605652
1  Random intercept + slope  NaN  NaN    -6355.672755
2       Classroom intercept  NaN  NaN    -8076.791887
